In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# theesis binary 15

In [1]:
# -*- coding: utf-8 -*-
# COMPLETE CODE
# DT FEATURE EXTRACTOR + FC / GRU CLASSIFIERS
# REMOVED:
# 1) GRU_h32_l1_fc32
# 2) GRU_h64_l1_fc32
# 3) GRU_h64_l2_fc64
# ADDED:
# 1) Trainable parameter counting
# 2) Total training time
# 3) Average epoch time
# 4) Total inference time
# 5) Inference time per sample

!python -m pip -q install torch torchvision torchaudio seaborn matplotlib scikit-learn pandas numpy

import os
os.environ["PYTHONHASHSEED"] = "42"

import copy
import random
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt


def set_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(42)

g = torch.Generator()
g.manual_seed(42)

#/kaggle/input/datasets/mohammedalaaraji/dataset44/NUSW-NB15_features.csv
# =========================================================
# 1) Configuration
# =========================================================
#PATH_CSV = "/kaggle/input/datasets/mohammednabeel10/dataset31/X-IIoTID.csv"
#TARGET_LABEL = "class1"  # CHANGED: was "class3"

BATCH_SIZE = 256
NUM_EPOCHS = 20
#LR = 0.0003
LR = 0.001
PATIENCE = 3

DEVICE_TYPE = "CPU"
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    DEVICE_TYPE = "TPU"
except Exception:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        DEVICE_TYPE = "GPU"
    else:
        device = torch.device("cpu")
        DEVICE_TYPE = "CPU"

print("Using device:", DEVICE_TYPE, device)
if DEVICE_TYPE == "GPU":
    print("GPU name:", torch.cuda.get_device_name(0))


# =========================================================
# 2) Safe Helpers
# =========================================================
def is_string_like(col):
    return pd.api.types.is_object_dtype(col) or pd.api.types.is_string_dtype(col)


def identify_column_types(df):
    datetime_cols = []
    categorical_cols = []
    numeric_cols = []

    for col in df.columns:
        if is_string_like(df[col]):
            if col.lower() in ["date", "timestamp"]:
                datetime_cols.append(col)
            else:
                sample = df[col].dropna().head(1000).astype(str)
                try:
                    pd.to_numeric(sample)
                    numeric_cols.append(col)
                except:
                    categorical_cols.append(col)
        else:
            numeric_cols.append(col)

    return datetime_cols, categorical_cols, numeric_cols


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

#TARGET_LABEL = "class3"




# =========================================================
# 3) Load Dataset
# =========================================================


print("Loading dataset...")

dfs = []
for i in range(1, 5):
    path = "/kaggle/input/datasets/drrabeashaker/dataset20/UNSW-NB15_{}.csv"
    dfs.append(pd.read_csv(path.format(i), header=None, low_memory=False))

df = pd.concat(dfs, axis=0).reset_index(drop=True)

# Load column names
# ===============================
df_col = pd.read_csv(
    "/kaggle/input/datasets/drrabeashaker/dataset20/NUSW-NB15_features.csv",
    encoding="ISO-8859-1"
)

# clean column names
df_col["Name"] = df_col["Name"].astype(str).apply(
    lambda x: x.strip().replace(" ", "").lower()
)

# assign column names
df.columns = df_col["Name"].values

#df.fillna(0, inplace=True)

print("Dataset shape:", df.shape)

# ✅ COUNT empty strings
num_empty = (df["attack_cat"] == "").sum()
print("Number of empty strings in attack_cat:", num_empty)

# ✅ ADD THESE TWO LINES HERE
df["attack_cat"] = df["attack_cat"].replace("", np.nan)


labels_df = df[["label", "attack_cat"]].copy()
features = df.drop(["label", "attack_cat"], axis=1)

X = features
y_raw = labels_df["label"]





# =========================================================
# 4) Train / Validation / Test Split
# =========================================================
X_temp, X_test, y_temp, y_test_raw = train_test_split(
    X, y_raw,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y_raw
)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_temp, y_temp,
    test_size=0.20,
    shuffle=True,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


# =========================================================
# 5) Label Encoding
# =========================================================
le = LabelEncoder()

y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

num_classes = len(le.classes_)
print("Classes:", le.classes_)


# =========================================================
# 6) Identify Column Types
# =========================================================
datetime_cols, categorical_cols, numeric_cols = identify_column_types(X_train)

print("Datetime:", len(datetime_cols))
print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))


# =========================================================
# 7) Copy Data
# =========================================================
X_train_p = X_train.copy()
X_val_p = X_val.copy()
X_test_p = X_test.copy()

label_encoders = {}

# ADD THIS BLOCK after step 7 (after copy data)

#for col in X_train.columns:
#    X_train_p[col + "_isnan"] = X_train[col].isna().astype(int)
#    X_val_p[col + "_isnan"]   = X_val[col].isna().astype(int)
#    X_test_p[col + "_isnan"]  = X_test[col].isna().astype(int)
# =========================================================
# 8) Handle Datetime Columns
# =========================================================
for col in datetime_cols:
    X_train_p[col] = pd.to_datetime(X_train_p[col], errors="coerce").astype("int64")
    X_val_p[col] = pd.to_datetime(X_val_p[col], errors="coerce").astype("int64")
    X_test_p[col] = pd.to_datetime(X_test_p[col], errors="coerce").astype("int64")


# =========================================================
# 9) Convert Numeric Object/String Columns
# =========================================================
for col in numeric_cols:
    if is_string_like(X_train_p[col]):
        X_train_p[col] = pd.to_numeric(X_train_p[col], errors="coerce")
        X_val_p[col] = pd.to_numeric(X_val_p[col], errors="coerce")
        X_test_p[col] = pd.to_numeric(X_test_p[col], errors="coerce")


# =========================================================
# 10) Encode Categorical Columns
# =========================================================
for col in categorical_cols:
    print("Processing:", col)

    le_col = LabelEncoder()



    train_str = X_train_p[col].fillna("MISSING").astype(str)
    val_str   = X_val_p[col].fillna("MISSING").astype(str)
    test_str  = X_test_p[col].fillna("MISSING").astype(str)

    le_col.fit(train_str)
    mapping = {cls: i for i, cls in enumerate(le_col.classes_)}

    X_train_p[col] = train_str.map(mapping).fillna(0).astype(int)
    X_val_p[col]   = val_str.map(mapping).fillna(0).astype(int)
    X_test_p[col]  = test_str.map(mapping).fillna(0).astype(int)

    label_encoders[col] = le_col


# =========================================================
# 11) Fill Missing Values + Scaling
# =========================================================
for col in X_train_p.columns:
    X_train_p[col] = pd.to_numeric(X_train_p[col], errors="coerce")
    X_val_p[col]   = pd.to_numeric(X_val_p[col], errors="coerce")
    X_test_p[col]  = pd.to_numeric(X_test_p[col], errors="coerce")

for col in numeric_cols:
    median_val = X_train_p[col].median()

    X_train_p[col] = X_train_p[col].fillna(median_val)
    X_val_p[col]   = X_val_p[col].fillna(median_val)
    X_test_p[col]  = X_test_p[col].fillna(median_val)




scaler = MinMaxScaler()

X_train_s = scaler.fit_transform(X_train_p)
X_val_s   = scaler.transform(X_val_p)
X_test_s  = scaler.transform(X_test_p)

print("Scaled shape:", X_train_s.shape)


# =========================================================
# 12) Train Decision Tree Feature Extractor
# =========================================================
print("Training Decision Tree...")

dt = DecisionTreeClassifier(
    ccp_alpha=0.0,
    class_weight=None,
    criterion='entropy',
    max_depth=12,
    max_features=None,
    min_samples_leaf=20,
    min_samples_split=2,
    splitter="best",
    random_state=42
)

dt.fit(X_train_s, y_train)

leaf_train = dt.apply(X_train_s).reshape(-1, 1)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

F_train = ohe.fit_transform(leaf_train)

print("Leaf features:", F_train.shape)

leaf_val  = dt.apply(X_val_s).reshape(-1, 1)
leaf_test = dt.apply(X_test_s).reshape(-1, 1)

F_val  = ohe.transform(leaf_val)
F_test = ohe.transform(leaf_test)

print("Original features:", X_train_s.shape[1])
print("DT features:", F_train.shape[1])


# =========================================================
# 13) PyTorch Dataset
# =========================================================
class IIoTIDDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = IIoTIDDataset(F_train, y_train)
val_dataset   = IIoTIDDataset(F_val, y_val)
test_dataset  = IIoTIDDataset(F_test, y_test)


# =========================================================
# 14) DataLoaders
# =========================================================
def make_loaders():
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=g,
        worker_init_fn=seed_worker
        
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        generator=g,
        worker_init_fn=seed_worker
        
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        generator=g,
        worker_init_fn=seed_worker
        
    )

    return train_loader, val_loader, test_loader


# =========================================================
# 15) Models
# =========================================================
class FCOnlyClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.fc(x)


class GRUSimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.0):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]
        out = self.fc(gru_out)
        return out


class GRUOneHiddenClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, fc_size, num_classes, dropout=0.0):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc1 = nn.Linear(hidden_size, fc_size)
        self.fc2 = nn.Linear(fc_size, num_classes)

        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]

        out = self.fc1(gru_out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out


# =========================================================
# 16) Model Configurations
# =========================================================
MODEL_CONFIGS = [
    {
        "name": "FC_only",
        "type": "fc_only"
    },
    {
        "name": "GRU_h8_l1_fc0",
        "type": "gru_simple",
        "hidden_size": 8,
        "num_layers": 1,
        "dropout": 0.0
    },
    {
        "name": "GRU_h16_l1_fc0",
        "type": "gru_simple",
        "hidden_size": 16,
        "num_layers": 1,
        "dropout": 0.0
    },
    {
        "name": "GRU_h32_l1_fc0",
        "type": "gru_simple",
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.0
    }
]


# =========================================================
# 17) Build Model
# =========================================================
def build_model(cfg, input_size, num_classes):
    if cfg["type"] == "fc_only":
        model = FCOnlyClassifier(
            input_size=input_size,
            num_classes=num_classes
        )
    elif cfg["type"] == "gru_simple":
        model = GRUSimpleClassifier(
            input_size=input_size,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            num_classes=num_classes,
            dropout=cfg["dropout"]
        )
    else:
        model = GRUOneHiddenClassifier(
            input_size=input_size,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            fc_size=cfg["fc_size"],
            num_classes=num_classes,
            dropout=cfg["dropout"]
        )

    return model.to(device)


# =========================================================
# 18) Train + Evaluate One Model
# =========================================================
def train_and_evaluate_model(cfg):
    print("\n" + "=" * 70)
    print("Training model:", cfg["name"])
    print(cfg)
    print("=" * 70)

    train_loader, val_loader, test_loader = make_loaders()

    model = build_model(cfg, F_train.shape[1], num_classes)

    num_params = count_trainable_parameters(model)
    print("Trainable parameters:", num_params)

    criterion  = nn.CrossEntropyLoss()
    optimizer  = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=1e-5
    )

    best_val_loss    = float("inf")
    best_val_acc     = 0.0
    best_epoch       = 0
    patience_counter = 0
    best_state       = None

    train_losses      = []
    val_losses        = []
    train_accuracies  = []
    val_accuracies    = []

    training_start_time = time.time()
    epoch_times         = []

    for epoch in range(NUM_EPOCHS):
        epoch_start_time = time.time()

        model.train()

        train_loss_sum = 0.0
        train_correct  = 0
        train_total    = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss_sum += loss.item() * y_batch.size(0)

            preds          = logits.argmax(dim=1)
            train_correct += (preds == y_batch).sum().item()
            train_total   += y_batch.size(0)

        train_epoch_loss = train_loss_sum / train_total
        train_acc        = 100.0 * train_correct / train_total

        model.eval()

        val_loss_sum = 0.0
        val_correct  = 0
        val_total    = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(X_batch)
                loss   = criterion(logits, y_batch)

                val_loss_sum += loss.item() * y_batch.size(0)

                preds        = logits.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total   += y_batch.size(0)

        val_epoch_loss = val_loss_sum / val_total
        val_acc        = 100.0 * val_correct / val_total

        train_losses.append(train_epoch_loss)
        val_losses.append(val_epoch_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        print(
            f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
            f"Train Loss: {train_epoch_loss:.6f} | "
            f"Val Loss: {val_epoch_loss:.6f} | "
            f"Train Acc: {train_acc:.4f}% | "
            f"Val Acc: {val_acc:.4f}%"
        )

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)
        print(f"Epoch Time: {epoch_time:.2f} sec")

        if val_epoch_loss < best_val_loss:
            best_val_loss    = val_epoch_loss
            best_val_acc     = val_acc
            best_epoch       = epoch + 1
            patience_counter = 0
            best_state       = copy.deepcopy(model.state_dict())
            print("Best model saved based on Val Loss")
        else:
            patience_counter += 1
            print(f"No improvement count: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

    total_training_time = time.time() - training_start_time
    avg_epoch_time      = sum(epoch_times) / len(epoch_times)

    model.load_state_dict(best_state)
    model.eval()

    preds       = []
    true_labels = []

    inference_start_time = time.time()

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits  = model(X_batch)
            p       = logits.argmax(dim=1).cpu().numpy()

            preds.append(p)
            true_labels.append(y_batch.numpy())

    preds       = np.concatenate(preds)
    true_labels = np.concatenate(true_labels)

    total_inference_time      = time.time() - inference_start_time
    inference_time_per_sample = total_inference_time / len(test_dataset)

    test_acc          = accuracy_score(true_labels, preds)
    test_macro_f1     = f1_score(true_labels, preds, average="macro")
    test_weighted_f1  = f1_score(true_labels, preds, average="weighted")

    print("\nTest Classification Report for", cfg["name"])
    target_names = [str(c) for c in le.classes_]

    print(classification_report(true_labels, preds, target_names=target_names, digits=4))
    

    result = {
        "name":                        cfg["name"],
        "num_params":                  num_params,
        "best_epoch":                  best_epoch,
        "best_val_loss":               best_val_loss,
        "best_val_acc":                best_val_acc,
        "test_acc":                    test_acc,
        "test_macro_f1":               test_macro_f1,
        "test_weighted_f1":            test_weighted_f1,
        "train_losses":                train_losses,
        "val_losses":                  val_losses,
        "train_accuracies":            train_accuracies,
        "val_accuracies":              val_accuracies,
        "total_training_time_sec":     total_training_time,
        "avg_epoch_time_sec":          avg_epoch_time,
        "total_inference_time_sec":    total_inference_time,
        "inference_time_per_sample_sec": inference_time_per_sample,
        "test_preds":                  preds,
        "test_true":                   true_labels
    }

    return result


# =========================================================
# 19) Train All Models
# =========================================================
all_results = []

for cfg in MODEL_CONFIGS:
    result = train_and_evaluate_model(cfg)
    all_results.append(result)


# =========================================================
# 20) Results Table
# =========================================================
results_df = pd.DataFrame([
    {
        "Model":                  r["name"],
        "Parameters":             r["num_params"],
        "Best Epoch":             r["best_epoch"],
        "Best Val Loss":          r["best_val_loss"],
        "Best Val Acc":           r["best_val_acc"],
        "Test Acc":               r["test_acc"],
        "Test Macro F1":          r["test_macro_f1"],
        "Test Weighted F1":       r["test_weighted_f1"],
        "Train Time (s)":         r["total_training_time_sec"],
        "Avg Epoch Time (s)":     r["avg_epoch_time_sec"],
        "Inference Time (s)":     r["total_inference_time_sec"],
        "Inference/sample (ms)":  r["inference_time_per_sample_sec"] * 1000
    }
    for r in all_results
])

results_df = results_df.sort_values(
    by=["Best Val Loss", "Test Macro F1"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))

Using device: GPU cuda
GPU name: Tesla T4
Loading dataset...
Dataset shape: (2540047, 49)
Number of empty strings in attack_cat: 0
Train: (1625629, 47)
Val: (406408, 47)
Test: (508010, 47)
Classes: [0 1]
Datetime: 0
Categorical: 6
Numeric: 41
Processing: srcip
Processing: dstip
Processing: proto
Processing: state
Processing: service
Processing: ct_ftp_cmd
Scaled shape: (1625629, 47)
Training Decision Tree...
Leaf features: (1625629, 261)
Original features: 47
DT features: 261

Training model: FC_only
{'name': 'FC_only', 'type': 'fc_only'}
Trainable parameters: 524
Epoch 01/20 | Train Loss: 0.086989 | Val Loss: 0.019845 | Train Acc: 98.6858% | Val Acc: 99.1814%
Epoch Time: 22.70 sec
Best model saved based on Val Loss
Epoch 02/20 | Train Loss: 0.014908 | Val Loss: 0.013221 | Train Acc: 99.3557% | Val Acc: 99.4004%
Epoch Time: 22.01 sec
Best model saved based on Val Loss
Epoch 03/20 | Train Loss: 0.012641 | Val Loss: 0.012749 | Train Acc: 99.4145% | Val Acc: 99.3991%
Epoch Time: 21.75 sec